<a href="https://colab.research.google.com/github/aimtyaem/AGO/blob/codespace-ideal-winner-j9j74wxj7q4hq7gq/CFPperCapita.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import xml.etree.ElementTree as ET
from collections import defaultdict
import re
import csv  # Added for CSV export

COLOR_MAP = {
    'Light Cream': {'hex': '#fff7ec', 'range': '0-2 tonnes'},
    'Pale Orange': {'hex': '#fee8c8', 'range': '2-4 tonnes'},
    'Light Tan': {'hex': '#fdd49e', 'range': '4-6 tonnes'},
    'Warm Beige': {'hex': '#fdbb84', 'range': '6-8 tonnes'},
    'Coral': {'hex': '#fc8d59', 'range': '8-10 tonnes'},
    'Brick Red': {'hex': '#ef6548', 'range': '10-15 tonnes'},
    'Fire Red': {'hex': '#d42b21', 'range': '15-20 tonnes'},
    'Dark Red': {'hex': '#990000', 'range': '20+ tonnes'},
    'White': {'hex': '#ffffff', 'range': 'No data'}
}

def parse_svg_colors(svg_file):
    tree = ET.parse(svg_file)
    root = tree.getroot()
    ns = {'svg': 'http://www.w3.org/2000/svg'}

    color_counts = defaultdict(int)

    for path in root.findall('.//svg:path', ns):
        fill = path.get('fill', '').lower()

        if fill.startswith('rgb('):
            numbers = list(map(int, re.findall(r'\d+', fill)))
            fill = "#{:02x}{:02x}{:02x}".format(*numbers)

        fill = re.sub(r'[^a-f0-9#]', '', fill)

        if fill and fill != "#aaa":
            for name, data in COLOR_MAP.items():
                if data['hex'].lower() == fill:
                    color_counts[name] += 1
                    break
            else:
                color_counts[fill] += 1

    return color_counts

# Process SVG file
svg_path = "CO2_emissions_per_capita,_2017_(Our_World_in_Data).svg"
color_counts = parse_svg_colors(svg_path)

# Print to console
print("CO₂ Emissions Classification:")
print("{:<20} {:<15} {}".format("Color Name", "Count", "Emission Range"))
print("-" * 45)
for color, count in color_counts.items():
    emission_range = COLOR_MAP[color]['range'] if color in COLOR_MAP else "Unknown"
    print(f"{color:<20} {count:<15} {emission_range}")  # Fixed typo here

# Save to CSV
csv_filename = "emissions_classification.csv"
with open(csv_filename, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['Color Name', 'Count', 'Emission Range'])

    for color, count in color_counts.items():
        emission_range = COLOR_MAP[color]['range'] if color in COLOR_MAP else "Unknown"
        writer.writerow([color, count, emission_range])

print(f"\nData saved to {csv_filename}")

CO₂ Emissions Classification:
Color Name           Count           Emission Range
---------------------------------------------
#002147              1               Unknown
e                    1               Unknown
#eee                 10              Unknown
Light Cream          56              0-2 tonnes
Pale Orange          42              2-4 tonnes
Dark Red             3               20+ tonnes
Light Tan            33              4-6 tonnes
#c51810              6               Unknown
Coral                17              8-10 tonnes
Warm Beige           22              6-8 tonnes
Brick Red            6               10-15 tonnes
#7f0000              3               Unknown
#d7301f              3               Unknown
#b30000              1               Unknown

Data saved to emissions_classification.csv
